# mas_topologies — ML Architecture Design Demo

This notebook walks through the full `mas_topologies` pipeline applied to a CNN design task for **CIFAR-10**.

**What you will see:**
1. How to define a `PromptSet` — the five role prompts that drive the multi-agent system
2. How to describe an ML problem as a `DesignInput`
3. How `prepare_context()` turns the problem into a text prompt the agents receive
4. How to run a single topology (baseline) and compare all five topologies
5. How to read the structured session log written after each run

**Five topologies available:**

| Topology | Agents involved | When to use |
|---|---|---|
| `single` | Architect only | Quick baseline |
| `peer_review` | Architect + Reviewer (iterative) | Catch code/design errors |
| `adversarial` | Architect + Adversary (iterative) | Push for better design |
| `parallel_ensemble` | N Architects + Judge | Diverse proposals |
| `self_reflection` | Architect self-critiques | Lightweight quality check |

> **Prerequisites:** `pip install -e .` from `mas_topologies_designer/`, or add `src/` to the path in the cell below.

## 1. Imports

In [2]:
import sys, os, json
from pathlib import Path
from IPython.display import Markdown, display

# If you have NOT run `pip install -e .` yet, uncomment the next line:
sys.path.insert(0, str(Path.cwd().parent / "src"))

from mas_topologies import TopologyAgent, PromptSet
from mas_topologies.tasks.ml_architect.models import DesignInput
from mas_topologies.tasks.ml_architect.context import prepare_context, parse_response

print("Imports OK")

Imports OK


## 2. API Setup

Set your API key in the environment before running this notebook:

```bash
export OPENAI_API_KEY="sk-..."
# or
export ANTHROPIC_API_KEY="sk-ant-..."
```

The cell below will raise a clear error if no key is found.

In [ ]:
import os

os.environ["OPENAI_API_KEY"]="sk-..."
# or
os.environ["ANTHROPIC_API_KEY"]="sk-..."


# Then call your function
#MODEL_CLIENT, PROVIDER_LABEL = build_client("auto")

In [4]:
def build_client(provider="auto"):
    """Return (model_client, label). Raises if no key is available."""
    if provider in ("openai", "auto") and os.environ.get("OPENAI_API_KEY"):
        from autogen_ext.models.openai import OpenAIChatCompletionClient
        model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
        client = OpenAIChatCompletionClient(model=model, api_key=os.environ["OPENAI_API_KEY"])
        return client, f"openai/{model}"

    if provider in ("anthropic", "auto") and os.environ.get("ANTHROPIC_API_KEY"):
        from autogen_ext.models.anthropic import AnthropicChatCompletionClient
        model = os.environ.get("ANTHROPIC_MODEL", "claude-haiku-4-5")
        client = AnthropicChatCompletionClient(model=model, api_key=os.environ["ANTHROPIC_API_KEY"])
        return client, f"anthropic/{model}"

    raise EnvironmentError(
        "No API key found. Export OPENAI_API_KEY or ANTHROPIC_API_KEY before running this notebook."
    )

# --- choose your provider here ---
MODEL_CLIENT, PROVIDER_LABEL = build_client("auto")   # "auto" | "openai" | "anthropic"

print(f"Provider : {PROVIDER_LABEL}")

Provider : openai/gpt-4o-mini


## 3. Define the Prompts

A `PromptSet` bundles the five role prompts that `TopologyAgent` distributes across agents depending on the chosen topology. Every field except `system_message` is optional — unused roles are simply ignored.

| Field | Role | Used by topology |
|---|---|---|
| `system_message` | Primary architect | all |
| `critic_message` | Peer reviewer | `peer_review` |
| `adversary_message` | Adversarial challenger | `adversarial` |
| `judge_message` | Final synthesiser | `parallel_ensemble` |
| `reflection_prompt` | Self-critique trigger | `self_reflection` |

The prompts below are written out in full so you can read, edit, and experiment with them directly.

In [5]:
# Build fence marker so literal ``` never appears in this source file
_F = chr(96) * 3

# ---------------------------------------------------------------------------
# Primary architect prompt — received by every topology's main agent
# ---------------------------------------------------------------------------
ARCHITECT_PROMPT = f"""You are an expert ML architect specialising in deep learning CNN design.
Your role is to analyse datasets and propose optimal neural network architectures.

==============================================================================
CRITICAL: INPUT SHAPE CONVENTION
==============================================================================
input_shape is always HWC (height, width, channels).
PyTorch models unpack it as:  H, W, C = input_shape
The first Conv2d layer MUST use C, never a hardcoded integer.

==============================================================================
YOUR ANALYSIS PROCESS
==============================================================================
1. Extract key properties — spatial dims, channels, dataset size, framework.
2. Incorporate feedback — when prior training results are provided, make targeted improvements.
3. Decide pooling upfront — state explicitly: "This architecture uses N pooling operations."
   MaxPool2d(2) halves each spatial dimension; reduction_factor = 2^N.
4. Calculate flatten size BEFORE writing nn.Linear():
     flatten_size = final_channels × (H // reduction_factor) × (W // reduction_factor)
   Use variables, never magic numbers (e.g. 128*(H//4)*(W//4), NOT 128*8*8).
5. Justify choices — capacity vs dataset size, regularisation strategy.
6. Specify hyperparameters — optimizer, learning_rate, batch_size, epochs,
   learning_rate_schedule, warmup_epochs, dropout_rate, weight_decay.

==============================================================================
REQUIRED OUTPUT FORMAT  (all three sections, in order)
==============================================================================

1. **Analysis Section** — dataset properties, design rationale, strategy.

2. **Implementation Section** — complete executable Python:

{_F}python
import torch, torch.nn as nn, torch.nn.functional as F

def create_model(input_shape, num_classes):
    H, W, C = input_shape
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(C, 32, kernel_size=3, padding=1)
            # ... more layers ...
            flatten_size = 128 * (H // 4) * (W // 4)   # 2 pooling ops
            self.fc_out = nn.Linear(flatten_size, num_classes)
        def forward(self, x):
            # add shape comments at key steps, e.g.: # [B, 32, H, W]
            return self.fc_out(x)   # raw logits — no softmax
    return Model()
{_F}

3. **Hyperparameters Section** — JSON (no comments, no trailing commas):

{_F}json
{{
  "hyperparameters": {{
    "optimizer": "adam",
    "learning_rate": 0.001,
    "weight_decay": 0.0,
    "batch_size": 64,
    "epochs": 20,
    "learning_rate_schedule": "none",
    "warmup_epochs": 0,
    "dropout_rate": 0.25
  }}
}}
{_F}
"""
print("ARCHITECT_PROMPT defined  ({} chars)".format(len(ARCHITECT_PROMPT)))

ARCHITECT_PROMPT defined  (2785 chars)


In [6]:
# ---------------------------------------------------------------------------
# Critic / Adversary / Judge / Reflection prompts
# ---------------------------------------------------------------------------

CRITIC_PROMPT = """You are an expert ML reviewer. Your role is to critique
neural network architectures proposed by the architect.

When reviewing, check:
- Tensor shape consistency (input → flatten → linear)
- Appropriate regularisation for the dataset size (BatchNorm, Dropout)
- Hyperparameter choices (learning rate, batch size, epochs)
- Code correctness: model must return RAW LOGITS, not softmax probabilities

Be constructive: acknowledge strengths, then list specific improvements."""

ADVERSARY_PROMPT = """You are an adversarial ML critic. Challenge the
architect's design aggressively:

- Demand justification for every architecture choice
- Point out failure modes, underfit/overfit risks
- Push for stronger regularisation or better capacity
- Highlight any hardcoded numbers that depend on input size
- Insist that the final model beats a simple baseline

Do NOT accept vague justifications. Force the architect to improve."""

JUDGE_PROMPT = """You are a senior ML engineering judge. Several architects
have independently proposed neural network designs for the same task.

Your job:
1. Read all proposals carefully.
2. Identify the strongest elements in each.
3. Synthesise a single best-of-breed design — do not just pick one proposal.
4. Return the full three-section format (Analysis / Implementation / Hyperparameters).
5. Explain which ideas you took from which proposal and why."""

REFLECTION_PROMPT = """Review your own architecture proposal above.

Check specifically:
1. Does every nn.Linear input size match the actual flattened tensor size?
2. Are Dropout layers placed correctly (after activations, before linear)?
3. Is the model returning raw logits (no softmax at the end)?
4. Are hyperparameters sensible for the stated dataset size?

If you find any issues, produce a corrected full proposal.
If everything looks correct, confirm and explain why."""

print("Role prompts defined")

Role prompts defined


In [7]:
# Assemble the five prompts into a PromptSet — the contract between
# your task definition and the topology machinery.
PROMPT_SET = PromptSet(
    system_message=ARCHITECT_PROMPT,
    critic_message=CRITIC_PROMPT,
    adversary_message=ADVERSARY_PROMPT,
    judge_message=JUDGE_PROMPT,
    reflection_prompt=REFLECTION_PROMPT,
)

print(PROMPT_SET)

PromptSet(system_message='You are an expert ML architect specialising in deep learning CNN design.\nYour role is to analyse datasets and propose optimal neural network architectures.\n\n==============================================================================\nCRITICAL: INPUT SHAPE CONVENTION\n==============================================================================\ninput_shape is always HWC (height, width, channels).\nPyTorch models unpack it as:  H, W, C = input_shape\nThe first Conv2d layer MUST use C, never a hardcoded integer.\n\n==============================================================================\nYOUR ANALYSIS PROCESS\n==============================================================================\n1. Extract key properties — spatial dims, channels, dataset size, framework.\n2. Incorporate feedback — when prior training results are provided, make targeted improvements.\n3. Decide pooling upfront — state explicitly: "This architecture uses N pooling operations

## 4. Define the ML Problem

`DesignInput` captures everything an ML architect needs to know: input shape, number of classes, dataset size, and any feedback from previous training runs.

`prepare_context()` serialises this into a readable text prompt. Print it to see exactly what the agents receive.

In [8]:
CIFAR10_INPUT = DesignInput(
    # --- dataset ---
    dataset_name="CIFAR-10",
    dataset_description=(
        "60 000 colour images (32×32, RGB) across 10 classes: airplane, automobile, bird, cat, "
        "deer, dog, frog, horse, ship, truck. 50 000 training images, 10 000 test images. "
        "Target: ≥92% validation accuracy."
    ),
    input_shape=(32, 32, 3),     # HWC convention
    num_classes=10,
    dataset_size=50_000,         # training samples
    # --- framework ---
    framework="pytorch",
    # --- feedback from a prior run (optional) ---
    previous_training_results=None,   # list of dicts, or None if first run
    user_notes=None,
)

# Show the agent's input prompt
context = prepare_context(CIFAR10_INPUT)
print(context)


# ML Architecture Design Task

## Dataset Information
- **Name**: CIFAR-10
- **Description**: 60 000 colour images (32×32, RGB) across 10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck. 50 000 training images, 10 000 test images. Target: ≥92% validation accuracy.
- **Number of Classes**: 10
- **Input Shape**: (32, 32, 3) (HWC format)
- **Dataset Size**: 50000 samples

## Framework & Environment
- **Framework**: PYTORCH
- **Version**: None

## Task
Analyze the above dataset properties and propose an optimal neural network architecture.

IMPORTANT:
- input_shape=(32, 32, 3) (HWC format) will be passed to create_model()
- The data_format_info shows the actual training data format for reference
- Design your architecture to work with the HWC input_shape format
- Provide implementation-ready code and specifications

Your response MUST include:
1. Analysis of dataset properties and design justifications
2. Complete Python code in a code block (see REQUIRED OUTP

## 5. Single-Agent Baseline

The simplest topology: one architect, no critique, one round.  
A good starting point — run this first to verify connectivity before trying multi-agent topologies.

Session logs are written to `logs/` alongside this notebook.

In [9]:
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

# TopologyAgent is the generic dispatcher — it reads the topology flag
# and routes to the appropriate multi-agent conversation.
agent_single = TopologyAgent(
    model_client=MODEL_CLIENT,
    topology="single",
    prompts=PROMPT_SET,
    log_dir=LOG_DIR,
)

# IPython/Jupyter supports top-level await — no asyncio.run() needed.
result_single = await agent_single.run(
    context,
    metadata={"dataset": CIFAR10_INPUT.dataset_name, "topology": "single"},
)

print(f"Rounds used : {result_single.rounds_used}")
print(f"Messages    : {len(result_single.messages)}")
print("─" * 60)
print(result_single.response_text[:2000], "..." if len(result_single.response_text) > 2000 else "")

[TopologyAgent] Session log: logs/session_20260606_180832.jsonl

[TopologyAgent] topology=single
1. **Analysis Section**  
The CIFAR-10 dataset consists of 60,000 color images of size 32x32 pixels, with 10 distinct classes. Given the small image size and 3 color channels (RGB), the model must be designed to efficiently learn patterns to achieve a validation accuracy target of ≥92%. The architecture needs to take into account the limited size of the dataset (50,000 training images) to avoid overfitting while maintaining sufficient capacity to learn effectively from the data.

To handle the images, I propose using a convolutional neural network (CNN) with several convolutional layers followed by pooling layers to progressively downsample the data and extract features, leading to a dense output layer for classification. The pooling operation will reduce spatial dimensions, allowing the network to focus on more abstract features while maintaining computational efficiency. This architecture

In [10]:
# parse_response() extracts structured fields from the free-text reply
output_single = parse_response(result_single.response_text, CIFAR10_INPUT)

display(Markdown(f"### Single-agent result\n\n**Hyperparameters:**\n```json\n{json.dumps(output_single.hyperparameters, indent=2)}\n```"))

### Single-agent result

**Hyperparameters:**
```json
{
  "framework": "pytorch",
  "input_shape": [
    32,
    32,
    3
  ],
  "num_classes": 10,
  "optimizer": "adam",
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "batch_size": 64,
  "epochs": 20,
  "learning_rate_schedule": "step",
  "warmup_epochs": 0,
  "dropout_rate": 0.25
}
```

## 6. Peer Review Topology

Now two agents participate: an **Architect** proposes a design, a **Reviewer** critiques it, and the Architect revises. This loop continues until the Reviewer is satisfied or `max_rounds` is reached.

Notice how the reviewer often catches shape bugs or weak regularisation that the single agent misses.

In [11]:
agent_peer = TopologyAgent(
    model_client=MODEL_CLIENT,
    topology="peer_review",
    prompts=PROMPT_SET,
    log_dir=LOG_DIR,
)

result_peer = await agent_peer.run(
    context,
    metadata={"dataset": CIFAR10_INPUT.dataset_name, "topology": "peer_review"},
)

print(f"Rounds used : {result_peer.rounds_used}")
print(f"Messages    : {len(result_peer.messages)}")
print("─" * 60)
print(result_peer.response_text[:2000], "..." if len(result_peer.response_text) > 2000 else "")

[TopologyAgent] Session log: logs/session_20260606_180902.jsonl

[TopologyAgent] topology=peer_review



--- Architect (round 1) ---
1. **Analysis Section**

The CIFAR-10 dataset consists of 60,000 images sized 32x32 pixels with 3 color channels (RGB), distributed among 10 classes. Given the small image size and the significant variability between classes (e.g., animals and vehicles), a convolutional neural network (CNN) is appropriate for this task.

**Key Properties**:
- **Spatial Dimensions**: 32x32
- **Channels**: 3 (RGB)
- **Number of Classes**: 10
- **Dataset Size**: 50,000 training images suggests the need for a model with sufficient capacity, but also requires regularization due to the risk of overfitting.

**Design Rationale**:
- **Convolutional Layers**: The model will use a series of convolutional layers to capture spatial hierarchies. We will apply layers with gradually increasing filters to learn more complex features.
- **Pooling**: The architecture will employ 2 MaxPooling operations to reduce spatial dimensions while maintaining important spatial hierarchies. This results

## 7. Compare All Five Topologies

Run all five topologies in sequence and collect a summary table.

> **Note:** `parallel_ensemble` and `adversarial` generate more tokens and may take a few minutes. Reduce `n_experts` to speed up `parallel_ensemble`.

In [12]:
from mas_topologies import VALID_TOPOLOGIES

results = {}   # topology -> TopologyResult

for topo in VALID_TOPOLOGIES:
    print(f"\n{'='*60}\nRunning topology: {topo}\n{'='*60}")
    agent = TopologyAgent(
        model_client=MODEL_CLIENT,
        topology=topo,
        prompts=PROMPT_SET,
        log_dir=LOG_DIR,
        n_experts=2,     # only relevant for parallel_ensemble
    )
    r = await agent.run(
        context,
        metadata={"dataset": CIFAR10_INPUT.dataset_name, "topology": topo},
    )
    results[topo] = r
    print(f"  rounds={r.rounds_used}  messages={len(r.messages)}")


Running topology: single
[TopologyAgent] Session log: logs/session_20260606_181118.jsonl

[TopologyAgent] topology=single
1. **Analysis Section**

The CIFAR-10 dataset contains 60,000 color images of size 32x32 across 10 distinct classes, making it a challenging benchmark for image classification. The images are relatively small and have three channels (RGB), which means the model must focus on learning spatial hierarchies effectively within a limited pixel space.

**Key considerations for architecture design:**
- **Input Shape**: The input dimension is (32, 32, 3); thus, the first convolutional layer will utilize 3 input channels, which are the RGB channels.
- **Number of Classes**: We have 10 output classes, necessitating a classification layer compatible with this number.
- **Dataset Size**: With 50,000 training samples, a model with moderate complexity should be adequate. This avoids overfitting while ensuring that the capacity is sufficient to learn the variety of representations

In [13]:
# Print a compact summary table
rows = ["| Topology | Rounds | Messages | Response length |",
        "|---|---|---|---|"]
for topo, r in results.items():
    rows.append(f"| `{topo}` | {r.rounds_used} | {len(r.messages)} | {len(r.response_text)} chars |")

display(Markdown("\n".join(rows)))

| Topology | Rounds | Messages | Response length |
|---|---|---|---|
| `single` | 1 | 1 | 3564 chars |
| `peer_review` | 4 | 7 | 4564 chars |
| `adversarial` | 4 | 7 | 6214 chars |
| `parallel_ensemble` | 1 | 3 | 10746 chars |
| `self_reflection` | 4 | 7 | 4059 chars |

## 8. Inspect the Session Logs

After each run, `TopologyAgent` writes a `.jsonl` file to `logs/`. Each line is one session. Load and display the most recent entry to see what was recorded.

In [14]:
log_files = sorted(LOG_DIR.glob("session_*.jsonl"))
print(f"Log files found: {len(log_files)}")

# Read the most recent log file
if log_files:
    latest = log_files[-1]
    print(f"\nLatest: {latest.name}\n{'─'*60}")
    with open(latest) as f:
        for i, line in enumerate(f):
            entry = json.loads(line)
            # Pretty-print everything except the full response text (can be long)
            short = {k: v for k, v in entry.items() if k != "response_text"}
            short["response_text_chars"] = len(entry.get("response_text", ""))
            print(f"Entry {i+1}:")
            print(json.dumps(short, indent=2, default=str))
            print()

Log files found: 24

Latest: session_20260606_181524.jsonl
────────────────────────────────────────────────────────────
Entry 1:
{
  "timestamp": "2026-06-06T18:16:46.127762",
  "topology": "self_reflection",
  "rounds_used": 4,
  "approved": false,
  "messages": [
    {
      "source": "Architect",
      "round": 1,
      "content": "1. **Analysis Section**  \nThe CIFAR-10 dataset consists of 60,000 color images of size 32x32 pixels across 10 classes. Since the images are small, the architecture should be compact yet effective enough to capture the relevant features for classification without overfitting. The input shape is (32, 32, 3), indicating 32 pixels in height, 32 pixels in width, and 3 color channels (RGB).\n\nGiven the dataset size of 50,000 training samples, a moderate-size convolutional neural network (CNN) would be appropriate to balance the capacity of the model and prevent overfitting. I propose an architecture that includes several convolutional layers followed by pooli

## 9. Display Full Architecture Proposal

Render the final response from any topology as formatted Markdown.

In [15]:
# Change this to any topology you want to inspect:
SHOW_TOPOLOGY = "peer_review"

r = results.get(SHOW_TOPOLOGY) or result_single

display(Markdown(f"### Full response — `{SHOW_TOPOLOGY}` topology\n\n---\n\n{r.response_text}"))

### Full response — `peer_review` topology

---

Thank you again for your detailed feedback and further suggestions! I appreciate the insights you've provided. Based on your recommendations, I’ll clarify the learning rate scheduler used, suggest implementation strategies for data augmentation, and finalize our architecture design with these enhancements. 

1. **Analysis Section - Adjustments**

### Enhanced Design Rationale:
1. **Learning Rate Scheduler**: We'll adopt a `StepLR` scheduler that reduces the learning rate by a factor (gamma) after a specified number of epochs (`step_size`). This structured approach helps in fine-tuning the model performance during the later stages of training.
2. **Data Augmentation**: Implementing data augmentation techniques in the preprocessing step will help increase dataset diversity and improve generalizability. This is critical given the relatively small size of the CIFAR-10 dataset.

---

2. **Implementation Section - Revised** 

Here’s the updated model definition including data augmentation and learning rate scheduling:

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch.optim as optim

def create_model(input_shape, num_classes):
    H, W, C = input_shape
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(C, 32, kernel_size=3, padding=1)  
            self.bn1 = nn.BatchNorm2d(32)                             
            self.pool1 = nn.MaxPool2d(2)                              
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) 
            self.bn2 = nn.BatchNorm2d(64)                             
            self.pool2 = nn.MaxPool2d(2)                              
            self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) 
            self.bn3 = nn.BatchNorm2d(128)                            
            self.dropout1 = nn.Dropout(0.25)
            self.dropout2 = nn.Dropout(0.5)  # Another dropout before final layer
            
            flatten_size = 128 * (H // 4) * (W // 4)  # 128 * 8 * 8 = 8192
            self.fc_out = nn.Linear(flatten_size, num_classes)
        
        def forward(self, x):
            x = self.pool1(F.relu(self.bn1(self.conv1(x))))  # [B, 32, 16, 16]
            x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # [B, 64, 8, 8]
            x = F.relu(self.bn3(self.conv3(x)))               # [B, 128, 8, 8]
            x = self.dropout1(x)
            x = x.view(x.size(0), -1)                         # Flatten the tensor
            x = self.dropout2(x)
            return self.fc_out(x)                             # raw logits — no softmax

    return Model()

# Create the data augmentation transformations
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])

# Assume `train_dataset` and `val_dataset` are your training and validation datasets
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, transform=transform)

# After defining your model
model = create_model(input_shape=(32, 32, 3), num_classes=10)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0005)

# StepLR Learning Rate Scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1) # Adjusting the lr every 10 epochs

# During training loop
for epoch in range(30):
    train(model, optimizer)  # Define your training function
    scheduler.step()  # Step the scheduler after each epoch
```

---

3. **Hyperparameters Section - Updated JSON**

```json
{
  "hyperparameters": {
    "optimizer": "adam",
    "learning_rate": 0.001,
    "weight_decay": 0.0005,
    "batch_size": 64,
    "epochs": 30,
    "learning_rate_schedule": "StepLR",  // Specify using the StepLR technique
    "step_size": 10,  // Learning rate will be adjusted every 10 epochs
    "gamma": 0.1,     // Reduce learning rate by a factor of 10
    "warmup_epochs": 3,
    "dropout_rate": 0.25
  }
}
```

---

### Conclusion

The integration of a learning rate scheduler and data augmentation techniques addresses your suggestions for optimizing the model’s training efficacy and robustness. This updated architecture design positions us well to achieve high predictive performance on the CIFAR-10 dataset, targeting the goal of ≥92% validation accuracy. I will continue to monitor metrics closely during training and make any adjustments necessary for optimal outcomes. Thank you for your continued guidance!

## 10. Executing the proposed architecture on CIFAR-10 data: 

In [16]:
import re

# parse_response() already extracted hyperparameters; now extract the Python code
code_pattern = r"```python\s*(.*?)\s*```"
matches = re.findall(code_pattern, r.response_text, re.DOTALL)

if not matches:
    raise ValueError("No python code block found in the agent response")

model_code = matches[-1]   # take the last (most revised) code block
print(model_code)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch.optim as optim

def create_model(input_shape, num_classes):
    H, W, C = input_shape
    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(C, 32, kernel_size=3, padding=1)  
            self.bn1 = nn.BatchNorm2d(32)                             
            self.pool1 = nn.MaxPool2d(2)                              
            self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) 
            self.bn2 = nn.BatchNorm2d(64)                             
            self.pool2 = nn.MaxPool2d(2)                              
            self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) 
            self.bn3 = nn.BatchNorm2d(128)                            
            self.dropout1 = nn.Dropout(0.25)
            self.dropout2 = nn.Dropout(0.5)  # Another dropout before final layer
          

In [17]:
exec(model_code, globals())   # defines create_model() in the notebook namespace
model = create_model(CIFAR10_INPUT.input_shape, CIFAR10_INPUT.num_classes)
print(model)

/home/axt807/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'train_dataset' is not defined

In [18]:
import torch
import torchvision, torchvision.transforms as T

hp = output_single.hyperparameters   # already parsed by parse_response()

transform = T.Compose([T.ToTensor(), T.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
train_set = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
val_set   = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=hp["batch_size"], shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=256)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = create_model(CIFAR10_INPUT.input_shape, CIFAR10_INPUT.num_classes).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=hp["learning_rate"], weight_decay=hp.get("weight_decay", 0))
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(hp["epochs"]):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        # CIFAR-10 from torchvision is BCHW — permute to BHWC if model expects HWC
        # (depends on whether the agent wrote the model in CHW or HWC convention)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    print(f"Epoch {epoch+1:3d}/{hp['epochs']}  val_acc={correct/total:.4f}")

100%|██████████| 170498071/170498071 [00:02<00:00, 68909646.20it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Epoch   1/20  val_acc=0.6159
Epoch   2/20  val_acc=0.7015
Epoch   3/20  val_acc=0.7248
Epoch   4/20  val_acc=0.7350
Epoch   5/20  val_acc=0.7679
Epoch   6/20  val_acc=0.7602
Epoch   7/20  val_acc=0.7608
Epoch   8/20  val_acc=0.7848
Epoch   9/20  val_acc=0.7874
Epoch  10/20  val_acc=0.7626
Epoch  11/20  val_acc=0.7742
Epoch  12/20  val_acc=0.7907
Epoch  13/20  val_acc=0.7994
Epoch  14/20  val_acc=0.7937
Epoch  15/20  val_acc=0.7921
Epoch  16/20  val_acc=0.7981
Epoch  17/20  val_acc=0.8071
Epoch  18/20  val_acc=0.7986
Epoch  19/20  val_acc=0.7995
Epoch  20/20  val_acc=0.8008


In [20]:
## Another 20 epochs: 
for epoch in range(hp["epochs"]):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        # CIFAR-10 from torchvision is BCHW — permute to BHWC if model expects HWC
        # (depends on whether the agent wrote the model in CHW or HWC convention)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    print(f"Epoch {epoch+1:3d}/{hp['epochs']}  val_acc={correct/total:.4f}")

Epoch   1/20  val_acc=0.8006
Epoch   2/20  val_acc=0.8069
Epoch   3/20  val_acc=0.7906
Epoch   4/20  val_acc=0.7975
Epoch   5/20  val_acc=0.8065
Epoch   6/20  val_acc=0.8034
Epoch   7/20  val_acc=0.8098
Epoch   8/20  val_acc=0.8094
Epoch   9/20  val_acc=0.7926
Epoch  10/20  val_acc=0.8087
Epoch  11/20  val_acc=0.8103
Epoch  12/20  val_acc=0.8008
Epoch  13/20  val_acc=0.8125
Epoch  14/20  val_acc=0.8077
Epoch  15/20  val_acc=0.8073
Epoch  16/20  val_acc=0.8098
Epoch  17/20  val_acc=0.8103
Epoch  18/20  val_acc=0.8109
Epoch  19/20  val_acc=0.8100
Epoch  20/20  val_acc=0.8136


### Claude-suggested training harness:

In [ ]:
"""
cifar10_training_harness.py
============================
A general-purpose training harness for AI-proposed CIFAR-10 models.

Usage in your notebook:
    1. exec(model_code, globals())          # defines create_model()
    2. run_training(hyperparams_json_str)   # does everything else
"""

import re
import json
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from dataclasses import dataclass


# ---------------------------------------------------------------------------
# 0.  CIFAR-10 input descriptor  (used by create_model() calls)
# ---------------------------------------------------------------------------

@dataclass
class CIFAR10Input:
    input_shape: tuple = (32, 32, 3)   # H, W, C
    num_classes: int   = 10

CIFAR10_INPUT = CIFAR10Input()


# ---------------------------------------------------------------------------
# 1.  Hyperparameter parsing  (strips JS-style // comments before JSON parse)
# ---------------------------------------------------------------------------

def parse_hyperparams(json_str: str) -> dict:
    """Parse hyperparameter JSON that may contain // comments."""
    clean = re.sub(r"//[^\n]*", "", json_str)   # strip // comments
    data  = json.loads(clean)
    # Support both {"hyperparameters": {...}} and bare {...}
    return data.get("hyperparameters", data)


# ---------------------------------------------------------------------------
# 2.  CIFAR-10 data loaders
# ---------------------------------------------------------------------------

# CIFAR-10 channel statistics (pre-computed on training split)
_CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
_CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

def get_cifar10_loaders(batch_size: int, data_root: str = "./data", num_workers: int = 2):
    """
    Download (if needed) and return (train_loader, val_loader).
    Augmentation is applied only to the training split.
    """
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(_CIFAR10_MEAN, _CIFAR10_STD),
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root=data_root, train=True,  download=True, transform=train_transform)
    val_dataset   = torchvision.datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)

    print(f"[data]  train={len(train_dataset):,}  val={len(val_dataset):,}  "
          f"batch_size={batch_size}")
    return train_loader, val_loader


# ---------------------------------------------------------------------------
# 3.  Optimizer factory  (general — add more branches as needed)
# ---------------------------------------------------------------------------

def build_optimizer(model: nn.Module, hp: dict) -> torch.optim.Optimizer:
    name = hp.get("optimizer", "adam").lower()
    lr   = float(hp.get("learning_rate", 1e-3))
    wd   = float(hp.get("weight_decay",  0.0))

    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    elif name in ("adamw",):
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif name == "sgd":
        momentum = float(hp.get("momentum", 0.9))
        return torch.optim.SGD(model.parameters(), lr=lr,
                               momentum=momentum, weight_decay=wd)
    elif name == "rmsprop":
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)
    else:
        raise ValueError(f"Unknown optimizer: '{name}'. "
                         "Add it to build_optimizer() or use adam/sgd/adamw.")


# ---------------------------------------------------------------------------
# 4.  LR scheduler factory  (general)
# ---------------------------------------------------------------------------

def build_scheduler(optimizer, hp: dict, steps_per_epoch: int):
    """
    Returns (scheduler, step_mode) where step_mode is 'epoch' or 'batch'.
    Call scheduler.step() after each epoch or each batch accordingly.
    """
    schedule = hp.get("learning_rate_schedule", "none").lower()

    if schedule in ("none", "constant", ""):
        return None, "epoch"

    elif schedule == "steplr":
        step_size = int(hp.get("step_size", 10))
        gamma     = float(hp.get("gamma", 0.1))
        return torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=step_size, gamma=gamma), "epoch"

    elif schedule == "cosineannealinglr":
        epochs = int(hp.get("epochs", 30))
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs), "epoch"

    elif schedule == "reducelronplateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", patience=5, factor=0.5), "epoch"

    elif schedule in ("onecyclelr", "one_cycle"):
        lr     = float(hp.get("learning_rate", 1e-3))
        epochs = int(hp.get("epochs", 30))
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr,
            steps_per_epoch=steps_per_epoch, epochs=epochs), "batch"

    elif schedule in ("warmup_cosine", "warmupcosine"):
        # Manual linear-warmup → cosine via LambdaLR
        epochs       = int(hp.get("epochs", 30))
        warmup_steps = int(hp.get("warmup_epochs", 3)) * steps_per_epoch
        total_steps  = epochs * steps_per_epoch
        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(warmup_steps, 1)
            progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
            import math
            return 0.5 * (1 + math.cos(math.pi * progress))
        return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda), "batch"

    else:
        print(f"[warn] Unknown schedule '{schedule}', skipping. "
              "Add it to build_scheduler() if needed.")
        return None, "epoch"


# ---------------------------------------------------------------------------
# 5.  Single train / eval epoch
# ---------------------------------------------------------------------------

def train_epoch(model, loader, optimizer, criterion, device, scheduler=None, step_mode="epoch"):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None and step_mode == "batch":
            scheduler.step()
        total_loss += loss.item() * inputs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += inputs.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs     = model(inputs)
            loss        = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += inputs.size(0)
    return total_loss / total, correct / total


# ---------------------------------------------------------------------------
# 6.  Main entry point
# ---------------------------------------------------------------------------

def run_training(hyperparams_json_str: str,
                 data_root:  str  = "./data",
                 num_workers: int = 2,
                 device_str: str  = "auto"):
    """
    Full training run.  Call after exec(model_code, globals()).

    Args:
        hyperparams_json_str : the raw JSON string (// comments are fine)
        data_root            : where to cache CIFAR-10
        num_workers          : DataLoader workers
        device_str           : 'auto' | 'cuda' | 'cpu'
    Returns:
        (model, history) where history is a list of per-epoch dicts
    """
    # --- device ---
    if device_str == "auto":
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_str)
    print(f"[device] {device}")

    # --- hyperparameters ---
    hp     = parse_hyperparams(hyperparams_json_str)
    epochs = int(hp.get("epochs", 30))
    bs     = int(hp.get("batch_size", 64))
    print(f"[hp]    {hp}")

    # --- data ---
    train_loader, val_loader = get_cifar10_loaders(bs, data_root, num_workers)

    # --- model  (create_model must be in globals() via exec) ---
    model = create_model(CIFAR10_INPUT.input_shape, CIFAR10_INPUT.num_classes)
    model = model.to(device)
    print(f"[model] {sum(p.numel() for p in model.parameters()):,} parameters")

    # --- optimizer / scheduler / loss ---
    optimizer           = build_optimizer(model, hp)
    scheduler, step_mode = build_scheduler(optimizer, hp, len(train_loader))
    criterion           = nn.CrossEntropyLoss()  # works with raw logits

    # --- training loop ---
    history = []
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(
            model, train_loader, optimizer, criterion, device, scheduler, step_mode)
        va_loss, va_acc = eval_epoch(
            model, val_loader, criterion, device)

        if scheduler is not None and step_mode == "epoch":
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_acc)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        row = dict(epoch=epoch,
                   train_loss=tr_loss, train_acc=tr_acc,
                   val_loss=va_loss,   val_acc=va_acc,
                   lr=current_lr)
        history.append(row)

        marker = " ◀ best" if va_acc > best_val_acc else ""
        best_val_acc = max(best_val_acc, va_acc)

        print(f"[{epoch:03d}/{epochs}] "
              f"train loss {tr_loss:.4f}  acc {tr_acc:.3f} | "
              f"val loss {va_loss:.4f}  acc {va_acc:.3f} | "
              f"lr {current_lr:.2e}{marker}")

    print(f"\n Training complete. Best val acc: {best_val_acc:.3f}")
    return model, history

In [ ]:
# =============================================================================
# CELL 1 — Imports & harness
# =============================================================================
import re, json
from cifar10_training_harness import CIFAR10_INPUT, run_training

# =============================================================================
# CELL 2 — Pick which topology result to train
#           (swap "peer_review" for any key in your `results` dict)
# =============================================================================
SHOW_TOPOLOGY = "peer_review"          # or "single", "hierarchical", etc.

r = results.get(SHOW_TOPOLOGY) or result_single   # your existing result object

# =============================================================================
# CELL 3 — Extract model code from the AI response
# =============================================================================
code_pattern = r"```python\s*(.*?)\s*```"
matches      = re.findall(code_pattern, r.response_text, re.DOTALL)

if not matches:
    raise ValueError("No python code block found in the agent response")

model_code = matches[-1]    # last (most revised) code block
print(model_code)

# =============================================================================
# CELL 4 — Register create_model() in the notebook namespace
# =============================================================================
exec(model_code, globals())     # defines create_model()

# Quick sanity check — make sure create_model is now available
assert callable(create_model), "create_model() was not defined by the agent code"
print("create_model() is registered")

# =============================================================================
# CELL 5 — Get hyperparameters as a JSON string
#
#   Two options depending on what your parse_response() returns:
#
#   Option A — already a dict  (e.g. output_single.hyperparameters)
#   Option B — raw JSON string still in r.response_text
# =============================================================================

# --- Option A: already a parsed dict ---
# hp_dict = r.hyperparameters          # or output_single.hyperparameters
# hyperparams_json_str = json.dumps(hp_dict)

# --- Option B: extract raw JSON block from response text (handles // comments) ---
json_pattern         = r"```json\s*(.*?)\s*```"
json_matches         = re.findall(json_pattern, r.response_text, re.DOTALL)

if json_matches:
    hyperparams_json_str = json_matches[-1]
    print("Hyperparameters extracted from response:\n", hyperparams_json_str)
elif hasattr(r, "hyperparameters") and r.hyperparameters:
    # Fall back to already-parsed dict on the result object
    hyperparams_json_str = json.dumps(r.hyperparameters)
    print("Hyperparameters from parsed result object:\n", hyperparams_json_str)
else:
    raise ValueError("Could not find hyperparameters — check your result object")

# =============================================================================
# CELL 6 — Train!
# =============================================================================
model, history = run_training(
    hyperparams_json_str,
    data_root   = "./data",   # where to cache CIFAR-10
    num_workers = 2,          # increase if your node has more CPU cores
    device_str  = "auto",     # "auto" picks GPU if available, else CPU
)

# =============================================================================
# CELL 7 (optional) — Quick results summary
# =============================================================================
best = max(history, key=lambda x: x["val_acc"])
print(f"\n{'='*50}")
print(f"Topology   : {SHOW_TOPOLOGY}")
print(f"Best epoch : {best['epoch']}")
print(f"Best val   : {best['val_acc']:.4f}")
print(f"Train acc  : {best['train_acc']:.4f}")
print(f"{'='*50}")

# =============================================================================
# CELL 8 (optional) — Compare multiple topologies in one shot
# =============================================================================
# Uncomment and run this cell instead of cells 2–7 if you want a side-by-side
# comparison of all topologies in your `results` dict.

# summary = {}
# for topology, r in results.items():
#     print(f"\n{'='*60}\nTraining topology: {topology}\n{'='*60}")
#
#     # --- extract model code ---
#     matches = re.findall(r"```python\s*(.*?)\s*```", r.response_text, re.DOTALL)
#     if not matches:
#         print(f"  [skip] no code block found for {topology}")
#         continue
#     exec(matches[-1], globals())
#
#     # --- extract hyperparams ---
#     json_matches = re.findall(r"```json\s*(.*?)\s*```", r.response_text, re.DOTALL)
#     hp_str = json_matches[-1] if json_matches else json.dumps(r.hyperparameters)
#
#     # --- train ---
#     _, history = run_training(hp_str, data_root="./data", device_str="auto")
#     best = max(history, key=lambda x: x["val_acc"])
#     summary[topology] = best
#
# print("\n\n===== TOPOLOGY COMPARISON =====")
# for topology, best in summary.items():
#     print(f"{topology:20s}  best_val={best['val_acc']:.4f}  epoch={best['epoch']}")